In [17]:
import ipywidgets as widgets
from IPython.display import display
import plotly.express as px
import os
import json
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [3]:
def charger_donnees_ucb(dossiers_ucb):
    """
    Parcourt les dossiers et extrait les métriques étape par étape.
    dossiers_ucb: dictionnaire de type {"Nom UCB": "chemin/vers/dossier"}
    """
    donnees = []
    
    for ucb_label, chemin_dossier in dossiers_ucb.items():
        if not os.path.exists(chemin_dossier):
            print(f"⚠️ Dossier introuvable : {chemin_dossier}")
            continue
            
        for nom_fichier in os.listdir(chemin_dossier):
            if nom_fichier.endswith(".json"):
                chemin_complet = os.path.join(chemin_dossier, nom_fichier)
                
                with open(chemin_complet, 'r') as f:
                    try:
                        exp = json.load(f)
                        
                        # Récupération de la graine
                        seed = exp.get("parameters", {}).get("seed", "Inconnu")
                        
                        # Accès aux 10 étapes du MCTS[cite: 2]
                        mcts_tries = exp.get("results", {}).get("mcts", {}).get("tries", [])
                        if not mcts_tries:
                            continue
                            
                        etapes = mcts_tries[0].get("steps", [])
                        
                        for index_etape, etape in enumerate(etapes):
                            metrics = etape.get("metrics", {})
                            
                            donnees.append({
                                "Constante_UCB": ucb_label,
                                "Seed": seed,
                                "Etape": index_etape + 1,
                                "Score": etape.get("score", 0), # Extraction du score[cite: 2]
                                "Temps_Etape_us": etape.get("stepTimeUs", 0),
                                # Extraction et conversion des métriques booléennes en 0/1[cite: 2]
                                "ParetoOptimal": int(metrics.get("ParetoOptimal", False)),
                                "EFX": int(metrics.get("EFX", False)),
                                "EF1": int(metrics.get("EF1", False)),
                                "EF": int(metrics.get("EF", False)),
                                "Prop": int(metrics.get("Prop", False))
                            })
                    except Exception as e:
                        print(f"Erreur de lecture sur {nom_fichier} : {e}")

    return pd.DataFrame(donnees)

In [ ]:
def tracer_comparaison_ucb_interactive(df, metrique_a_tracer="Score"):
    if df.empty:
        print("Aucune donnée à tracer.")
        return

    # 1. Récupération des seeds uniques disponibles dans les données
    seeds_disponibles = sorted(df['Seed'].unique().tolist())
    
    # 2. Création du menu déroulant pour la Seed
    dropdown_seed = widgets.Dropdown(
        options=seeds_disponibles,
        value=seeds_disponibles[0], # Valeur par défaut
        description='🌱 Seed:',
        layout={'width': 'max-content'}
    )
    
    out = widgets.Output()

    def update_plot(change=None):
        with out:
            out.clear_output(wait=True)
            seed_selectionnee = dropdown_seed.value
            
            # Filtrage : on ne garde que les lignes correspondant à la seed choisie
            df_filtre = df[df['Seed'] == seed_selectionnee]
            
            if df_filtre.empty:
                print(f"⚠️ Aucune donnée trouvée pour la seed {seed_selectionnee}.")
                return

            # Création du graphique avec les données brutes de la seed
            fig = px.line(
                df_filtre,
                x="Etape",
                y=metrique_a_tracer,
                color="Constante_UCB",
                markers=True,
                title=f"Évolution de la métrique '{metrique_a_tracer}' — <b>Seed : {seed_selectionnee}</b>",
                labels={
                    "Etape": "Étape (1 à 10)",
                    metrique_a_tracer: metrique_a_tracer,
                    "Constante_UCB": "Constante UCB (c)"
                }
            )
            
            # Forcer l'axe X à afficher les étapes sous forme d'entiers (1, 2, 3...)
            fig.update_xaxes(tickmode='linear', tick0=1, dtick=1)
            
            # Mise en forme
            fig.update_layout(
                template="plotly_white", 
                hovermode="x unified",
                margin=dict(t=60, b=40, l=40, r=40)
            )
            
            fig.show()

    # 3. Lier l'événement de changement du menu au rafraîchissement du graphique
    dropdown_seed.observe(update_plot, names='value')
    
    # 4. Afficher l'interface
    display(dropdown_seed, out)
    update_plot()

In [ ]:
def tracer_comparaison_metriques_interactive(df):
    if df.empty:
        print("Aucune donnée à tracer.")
        return

    # Liste des métriques à analyser
    colonnes_metriques = ['ParetoOptimal', 'EFX', 'EF1', 'EF', 'Prop']
    metriques_presentes = [m for m in colonnes_metriques if m in df.columns]

    # 1. Récupération des seeds disponibles
    seeds_disponibles = sorted(df['Seed'].unique().tolist())
    
    dropdown_seed = widgets.Dropdown(
        options=seeds_disponibles,
        value=seeds_disponibles[0],
        description='🌱 Seed:',
        layout={'width': 'max-content'}
    )
    
    out = widgets.Output()

    def update_plot(change=None):
        with out:
            out.clear_output(wait=True)
            seed_selectionnee = dropdown_seed.value
            
            # Filtrage sur la seed
            df_filtre = df[df['Seed'] == seed_selectionnee]
            if df_filtre.empty:
                print(f"⚠️ Aucune donnée trouvée pour la seed {seed_selectionnee}.")
                return

            # --- TRANSFORMATION DES DONNÉES ---
            # On regroupe toutes les colonnes de métriques dans deux colonnes : "Metrique" et "Valeur"
            df_melt = df_filtre.melt(
                id_vars=['Constante_UCB', 'Etape'],
                value_vars=metriques_presentes,
                var_name='Metrique',
                value_name='Valeur'
            )

            # --- ASTUCE ANTI-SUPERPOSITION ---
            # On applique un micro-décalage Y (-0.04, 0, +0.04) selon l'UCB pour séparer visuellement les lignes
            ucbs_uniques = sorted(df_melt['Constante_UCB'].unique())
            offsets = {ucb: (i - (len(ucbs_uniques) - 1) / 2.0) * 0.04 for i, ucb in enumerate(ucbs_uniques)}
            df_melt['Valeur_Affichee'] = df_melt.apply(lambda r: r['Valeur'] + offsets[r['Constante_UCB']], axis=1)

            # --- CRÉATION DU GRAPHIQUE ---
            fig = px.line(
                df_melt,
                x="Etape",
                y="Valeur_Affichee",
                color="Metrique",           # Couleur selon la métrique
                symbol="Metrique",          # Forme du point selon la métrique
                line_dash="Constante_UCB",  # Type de ligne (pleine/pointillée) selon l'UCB
                markers=True,
                title=f"Atteinte des Métriques d'Équité — <b>Seed : {seed_selectionnee}</b>",
                labels={
                    "Etape": "Étape du MCTS (1 à 10)",
                    "Valeur_Affichee": "Statut de la Métrique",
                    "Metrique": "Métrique",
                    "Constante_UCB": "Constante UCB (c)"
                },
                hover_data={
                    "Valeur_Affichee": False, # Cache le faux score décalé dans l'infobulle
                    "Valeur": True,           # Affiche la vraie valeur (0 ou 1)
                    "Constante_UCB": True,
                    "Metrique": True,
                    "Etape": True
                }
            )
            
            # Forcer l'affichage de l'axe X par nombres entiers (étapes 1, 2, 3...)
            fig.update_xaxes(tickmode='linear', tick0=1, dtick=1)
            
            # Forcer l'axe Y à afficher proprement "0" et "1"
            fig.update_yaxes(
                tickvals=[0, 1], 
                ticktext=["0 (Échec)", "1 (Atteint)"],
                range=[-0.15, 1.15] # Donne un peu de marge pour les lignes décalées
            )
            
            # Esthétique générale
            fig.update_layout(
                template="plotly_white", 
                hovermode="x unified",
                margin=dict(t=60, b=40, l=40, r=40)
            )
            
            # Élargir un peu les points et les lignes
            fig.update_traces(line=dict(width=2.5), marker=dict(size=9))

            fig.show()

    # Lier l'événement du menu au graphique
    dropdown_seed.observe(update_plot, names='value')
    
    # Affichage
    display(dropdown_seed, out)
    update_plot()

In [19]:
def tracer_comparaison_epuree_interactive(df):
    if df.empty:
        print("Aucune donnée à tracer.")
        return

    # 1. Identifier les colonnes disponibles
    colonnes_possibles = ['Score', 'ParetoOptimal', 'EFX', 'EF1', 'EF', 'Prop']
    metriques_presentes = [m for m in colonnes_possibles if m in df.columns]

    # 2. Widgets : Slider (Seed + Moyenne) et Cases à cocher (Métriques)
    seeds_disponibles = sorted(df['Seed'].unique().tolist())
    options_slider = ['Moyenne'] + seeds_disponibles
    
    slider_seed = widgets.SelectionSlider(
        options=options_slider,
        value='Moyenne', # Valeur par défaut
        description='🌱 Seed:',
        continuous_update=False,
        layout={'width': '500px'}
    )
    
    # Création des cases à cocher pour la sélection multiple
    checkboxes_metriques = [
        widgets.Checkbox(value=(m == 'Score'), description=m, layout={'width': 'max-content'})
        for m in metriques_presentes
    ]
    boite_metriques = widgets.HBox([widgets.Label("📊 Analyser :")] + checkboxes_metriques)
    
    ui = widgets.VBox([slider_seed, boite_metriques])
    out = widgets.Output()

    def update_plot(change=None):
        with out:
            out.clear_output(wait=True)
            seed_sel = slider_seed.value
            
            # Récupération de toutes les métriques actuellement cochées
            metriques_sel = [cb.description for cb in checkboxes_metriques if cb.value]
            
            if not metriques_sel:
                print("⚠️ Veuillez cocher au moins une métrique à afficher.")
                return

            # --- GESTION DES DONNÉES (Moyenne ou Graine spécifique) ---
            if seed_sel == 'Moyenne':
                # On fait la moyenne de toutes les valeurs numériques pour chaque étape et chaque UCB
                df_filtre = df.groupby(['Constante_UCB', 'Etape']).mean(numeric_only=True).reset_index()
                titre_seed = "Moyenne sur toutes les graines"
            else:
                df_filtre = df[df['Seed'] == seed_sel].copy()
                titre_seed = f"Graine (Seed): {seed_sel}"
            
            if df_filtre.empty:
                print(f"⚠️ Aucune donnée pour la sélection actuelle.")
                return

            # --- CONFIGURATION DU GRAPHIQUE ---
            fig = make_subplots(specs=[[{"secondary_y": True}]])
            ucbs_uniques = sorted(df_filtre['Constante_UCB'].unique())
            
            # Couleurs (1 couleur = 1 UCB) et styles de traits (1 style = 1 Métrique)
            couleurs = px.colors.qualitative.Set1
            map_couleurs = {ucb: couleurs[i % len(couleurs)] for i, ucb in enumerate(ucbs_uniques)}
            dash_styles = ['solid', 'dash', 'dot', 'dashdot', 'longdash']
            map_dash = {m: dash_styles[i % len(dash_styles)] for i, m in enumerate(metriques_sel)}

            # --- DESSIN DES COURBES ---
            for ucb in ucbs_uniques:
                df_ucb = df_filtre[df_filtre['Constante_UCB'] == ucb].sort_values('Etape')
                
                for metrique in metriques_sel:
                    is_score = (metrique == 'Score')
                    y_vals = df_ucb[metrique]
                    
                    # Décalage anti-superposition (uniquement pour les métriques d'équité, pas pour le Score)
                    if not is_score:
                        offset = (ucbs_uniques.index(ucb) - (len(ucbs_uniques) - 1) / 2.0) * 0.03
                        y_plot = y_vals + offset
                    else:
                        y_plot = y_vals

                    # Le "Score" est toujours disponible dans le DataFrame car on garde au moins cette colonne
                    score_vals = df_ucb['Score'] if 'Score' in df_ucb.columns else [0]*len(df_ucb)
                    
                    # Ajout des données de survol : valeur de la métrique ET Score (brut ou moyen)
                    custom_data = list(zip(y_vals, score_vals))
                    
                    hover_text = (
                        "<b>Étape %{x}</b><br>"
                        f"UCB : <b>{ucb}</b><br>"
                        f"Métrique (<b>{metrique}</b>) : %{{customdata[0]:.2f}}<br>"
                        f"Score MCTS {'moyen ' if seed_sel == 'Moyenne' else ''}: <b>%{{customdata[1]:.2f}}</b>"
                    )
                    
                    fig.add_trace(
                        go.Scatter(
                            x=df_ucb['Etape'],
                            y=y_plot,
                            mode='lines+markers',
                            name=f"{ucb} - {metrique}",
                            line=dict(color=map_couleurs[ucb], dash=map_dash[metrique], width=2.5),
                            marker=dict(size=8),
                            customdata=custom_data,
                            hovertemplate=hover_text,
                            legendgroup=ucb,
                            legendgrouptitle_text=ucb if metrique == metriques_sel[0] else None
                        ),
                        secondary_y=is_score
                    )
            
            # --- ESTHÉTIQUE DES AXES ---
            fig.update_xaxes(title_text="Étape du MCTS (1 à 10)", tickmode='linear', tick0=1, dtick=1)
            
            # Axe Y Primaire (Gauche : Métriques Binaires / Taux de réussite)
            if any(m != 'Score' for m in metriques_sel):
                if seed_sel == 'Moyenne':
                    titre_axe_gauche = "Taux de réussite (Moyenne)"
                    ticks_text = ["0% (Échec total)", "50%", "100% (Atteint partout)"]
                    ticks_vals = [0, 0.5, 1]
                else:
                    titre_axe_gauche = "Statut (0 = Échec, 1 = Atteint)"
                    ticks_text = ["0 (Échec)", "1 (Atteint)"]
                    ticks_vals = [0, 1]
                
                fig.update_yaxes(
                    title_text=titre_axe_gauche, 
                    tickvals=ticks_vals, 
                    ticktext=ticks_text,
                    range=[-0.1, 1.1], 
                    secondary_y=False
                )
            else:
                fig.update_yaxes(showticklabels=False, secondary_y=False)
            
            # Axe Y Secondaire (Droite : Score)
            if 'Score' in metriques_sel:
                fig.update_yaxes(title_text="Score Absolu", secondary_y=True)
            
            fig.update_layout(
                title=f"Comparaison Multi-Métriques des UCB — <b>{titre_seed}</b>",
                template="plotly_white", 
                hovermode="x unified",
                margin=dict(t=60, b=40, l=40, r=60)
            )
            fig.show()

    # 3. Écouteurs d'événements
    slider_seed.observe(update_plot, names='value')
    for cb in checkboxes_metriques:
        cb.observe(update_plot, names='value')
    
    # 4. Affichage
    display(ui, out)
    update_plot()

In [20]:
df_ucb = charger_donnees_ucb({
    "c = -20": "../resultsLowExplorationConstant/experiments_21-07-2026_11-51-51",
    "c = sqrt(2)": "../resultsSQRT2Constant/experiments_22-07-2026_09-09-49",
    "c = 20": "../resultsHighExplorationConstant/experiments_21-07-2026_09-37-37"
})

""" tracer_comparaison_ucb_interactive(df_ucb, metrique_a_tracer="Score")
tracer_comparaison_metriques_interactive(df_ucb) """
tracer_comparaison_epuree_interactive(df_ucb)

Output()